# 01 - Main Experiments (Two-Step + Main Comparison)

Mục tiêu notebook này:
- Tái hiện thực nghiệm chính trên CIFAR-10 theo hướng two-step (feature extractor + clustering net).
- So sánh với baseline cố định K để minh họa hiện tượng sụp hiệu năng khi K sai.
- Đối chiếu ACC với số liệu tham chiếu trong paper (Table 5, K0=30).


In [ ]:
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

sys.path.append('..')
from src.utils import set_seed, get_device, extract_features, train_clustering

set_seed(42)
device = get_device()
print(f'[INFO] device = {device}')


In [ ]:
# Two-step: bước 1 = trích xuất đặc trưng
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device)

transform_cifar = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

cifar_test = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_cifar)
subset = torch.utils.data.Subset(cifar_test, range(3000))
loader = DataLoader(subset, batch_size=128, shuffle=False)

X, y = extract_features(loader, backbone, device)
print('[INFO] features:', X.shape, '| labels:', y.shape)


In [ ]:
def run_setting(name, k0, enable_split, enable_merge, lambda_param=2.0, epochs=30, lr=1e-3):
    k_star, acc, nmi, ari = train_clustering(
        features=X,
        labels=y,
        k_max=k0,
        device=device,
        epochs=epochs,
        lr=lr,
        lambda_param=lambda_param,
        enable_split=enable_split,
        enable_merge=enable_merge,
    )
    return {
        'Setting': name,
        'K0': k0,
        'Split': enable_split,
        'Merge': enable_merge,
        'K*': int(k_star),
        'ACC(%)': acc * 100,
        'NMI(%)': nmi * 100,
        'ARI(%)': ari * 100,
    }


In [ ]:
results = []
results.append(run_setting('PnP (dynamic K)', k0=30, enable_split=True, enable_merge=True))
results.append(run_setting('Fixed-K baseline (wrong K=30)', k0=30, enable_split=False, enable_merge=False))
results.append(run_setting('Fixed-K baseline (oracle K=10)', k0=10, enable_split=False, enable_merge=False))

main_df = pd.DataFrame(results)
main_df


## Đối chiếu với paper (ACC, Table 5)

Trong paper *Deep Plug-and-Play Clustering*, bảng theo K0 cho CIFAR-10 có ACC tham chiếu:
- SCAN tại K0=30: **33.4**
- Ours tại K0=30: **82.0**

Bên dưới là độ lệch tương đối ACC giữa bản tái hiện của nhóm và số liệu tham chiếu.


In [ ]:
paper_acc = {
    'PnP (dynamic K)': 82.0,
    'Fixed-K baseline (wrong K=30)': 33.4,
}

cmp_rows = []
for _, row in main_df.iterrows():
    setting = row['Setting']
    if setting in paper_acc:
        reported = paper_acc[setting]
        reproduced = row['ACC(%)']
        rel_dev = abs(reproduced - reported) / max(1e-8, reported) * 100
        cmp_rows.append({
            'Setting': setting,
            'Paper ACC(%)': reported,
            'Reproduced ACC(%)': reproduced,
            'Relative Deviation(%)': rel_dev,
        })

cmp_df = pd.DataFrame(cmp_rows)
cmp_df


## Phân tích sai lệch (điền vào báo cáo)

Gợi ý phân tích theo rubric:
- Dùng ResNet18 pretrained ImageNet + subset 3000 mẫu, khác backbone/thiết lập paper.
- Cài đặt PnP ở mức học thuật (re-implementation), không dùng code gốc và chưa tune siêu tham số sâu.
- Khác biệt số vòng lặp, chuẩn hóa ảnh, optimizer và seed có thể gây sai lệch đáng kể.
- Dù vậy, xu hướng chính vẫn kiểm chứng được: mô hình fixed-K nhạy với K sai, còn PnP động ổn định hơn.
